# 02 Baseline Preprocessing

This notebook prepares the shared baseline dataset for multiclass intrusion detection.

Main purpose:
- Combine UNSW-NB15_1.csv ~ UNSW-NB15_4.csv
- Assign feature names
- Clean basic missing/unknown values
- Remove leakage and identifier columns
- Create a fixed stratified train/test split
- Save processed train/test data locally in `data/processed/`

Target:
- `attack_cat`

Classes:
- Normal
- Generic
- Exploits
- Fuzzers
- DoS
- Reconnaissance
- Analysis
- Backdoors
- Shellcode
- Worms

In [2]:
# import and path setting

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data folder:", RAW_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())

Project root: C:\Projects\nid-data-mining
Raw data folder: C:\Projects\nid-data-mining\data\raw
Processed folder: C:\Projects\nid-data-mining\data\processed


In [3]:
# Check raw files` existence
required_files = [
    "UNSW-NB15_1.csv",
    "UNSW-NB15_2.csv",
    "UNSW-NB15_3.csv",
    "UNSW-NB15_4.csv",
    "NUSW-NB15_features.csv"
]

for file in required_files:
    path = RAW_DIR / file
    print(file, "FOUND" if path.exists() else "MISSING")

UNSW-NB15_1.csv FOUND
UNSW-NB15_2.csv FOUND
UNSW-NB15_3.csv FOUND
UNSW-NB15_4.csv FOUND
NUSW-NB15_features.csv FOUND


In [4]:
# Load feature names
features_path = RAW_DIR / "NUSW-NB15_features.csv"

features = pd.read_csv(features_path, encoding="latin1")

features.head()
features.columns

columns = features["Name"].str.strip().str.lower().tolist()

print(len(columns))
print(columns[:10])
print(columns[-5:])

49
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl']
['ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'label']


In [5]:
# Load and combine the four CSVs
files = [
    RAW_DIR / "UNSW-NB15_1.csv",
    RAW_DIR / "UNSW-NB15_2.csv",
    RAW_DIR / "UNSW-NB15_3.csv",
    RAW_DIR / "UNSW-NB15_4.csv",
]

df_list = []

for file in files:
    print(f"Loading {file.name}...")
    temp = pd.read_csv(
        file,
        header=None,
        names=columns,
        low_memory=False
    )
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

df.columns = df.columns.str.strip().str.lower()

print("Shape:", df.shape)
df.head()

Loading UNSW-NB15_1.csv...
Loading UNSW-NB15_2.csv...
Loading UNSW-NB15_3.csv...
Loading UNSW-NB15_4.csv...
Shape: (2540047, 49)


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,label
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,...,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,...,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,...,0,7,9,1,1,1,1,1,NaN,0


In [6]:
# Basic data check

df.info()

df[["attack_cat", "label"]].head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2540047 entries, 0 to 2540046
Data columns (total 49 columns):
 #   Column            Dtype  
---  ------            -----  
 0   srcip             object 
 1   sport             object 
 2   dstip             object 
 3   dsport            object 
 4   proto             object 
 5   state             object 
 6   dur               float64
 7   sbytes            int64  
 8   dbytes            int64  
 9   sttl              int64  
 10  dttl              int64  
 11  sloss             int64  
 12  dloss             int64  
 13  service           object 
 14  sload             float64
 15  dload             float64
 16  spkts             int64  
 17  dpkts             int64  
 18  swin              int64  
 19  dwin              int64  
 20  stcpb             int64  
 21  dtcpb             int64  
 22  smeansz           int64  
 23  dmeansz           int64  
 24  trans_depth       int64  
 25  res_bdy_len       int64  
 26  sjit          

,attack_cat,label
0,NaN,0
1,NaN,0
2,NaN,0
3,NaN,0
4,NaN,0
5,NaN,0
6,NaN,0
7,NaN,0
8,NaN,0
9,NaN,0


In [7]:
# Normal traffic may have missing attack_cat in the full CSV files
df["attack_cat"] = df["attack_cat"].fillna("Normal")
df["attack_cat"] = df["attack_cat"].astype(str).str.strip()

# service "-" means unidentified service
df["service"] = df["service"].replace("-", "unknown")
df["service"] = df["service"].fillna("unknown")
df["service"] = df["service"].astype(str).str.strip()

df[["attack_cat", "label"]].head(20)

,attack_cat,label
0,Normal,0
1,Normal,0
2,Normal,0
3,Normal,0
4,Normal,0
5,Normal,0
6,Normal,0
7,Normal,0
8,Normal,0
9,Normal,0


In [8]:
# Check attack_cat and label relationship

pd.crosstab(df["attack_cat"], df["label"], dropna=False)

label,0,1
attack_cat,,
Analysis,0,2677
Backdoor,0,1795
Backdoors,0,534
DoS,0,16353
Exploits,0,44525
Fuzzers,0,24246
Generic,0,215481
Normal,2218764,0
Reconnaissance,0,13987


In [9]:
# Define target and drop leakage columns
target_col = "attack_cat"

drop_cols = [
    "label",       # target
    "attack_cat",  # leakage for binary classification
    "srcip",       # identifier, may cause memorization
    "dstip",       # identifier, may cause memorization
    "stime",       # timestamp, may cause dataset-specific artifacts
    "ltime"        # timestamp, may cause dataset-specific artifacts
]

drop_cols = [col for col in drop_cols if col in df.columns]

X = df.drop(columns=drop_cols)
y = df[target_col]

print("Dropped columns:", drop_cols)
print("X shape:", X.shape)
print("y shape:", y.shape)

Dropped columns: ['label', 'attack_cat', 'srcip', 'dstip', 'stime', 'ltime']
X shape: (2540047, 43)
y shape: (2540047,)


In [10]:
# Check categorical and numeric columns
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

print("\nNumber of categorical columns:", len(categorical_cols))
print("Number of numeric columns:", len(numeric_cols))
print("First 10 numeric columns:", numeric_cols[:10])

Categorical columns:
['sport', 'dsport', 'proto', 'state', 'service', 'ct_ftp_cmd']

Number of categorical columns: 6
Number of numeric columns: 37
First 10 numeric columns: ['dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'sload', 'dload', 'spkts']


In [11]:
# baseline missing value handling
# Fill categorical missing values with "unknown"
for col in categorical_cols:
    X[col] = X[col].fillna("unknown").astype(str).str.strip()

# Convert numeric columns safely and fill missing values with median
for col in numeric_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X[col] = X[col].fillna(X[col].median())

X.isnull().sum().sort_values(ascending=False).head(10)

sport              0
ct_state_ttl       0
sjit               0
djit               0
sintpkt            0
dintpkt            0
tcprtt             0
synack             0
ackdat             0
is_sm_ips_ports    0
dtype: int64

In [12]:
# train/test stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (2032037, 43)
X_test: (508010, 43)
y_train: (2032037,)
y_test: (508010,)


In [13]:
# Verify class distribution after split
train_dist = pd.DataFrame({
    "Train Count": y_train.value_counts(),
    "Train Percentage": (y_train.value_counts(normalize=True) * 100).round(2)
})

test_dist = pd.DataFrame({
    "Test Count": y_test.value_counts(),
    "Test Percentage": (y_test.value_counts(normalize=True) * 100).round(2)
})

split_distribution = train_dist.join(test_dist)

split_distribution

,Train Count,Train Percentage,Test Count,Test Percentage
attack_cat,,,,
Normal,1775011,87.35,443753,87.35
Generic,172385,8.48,43096,8.48
Exploits,35620,1.75,8905,1.75
Fuzzers,19397,0.95,4849,0.95
DoS,13082,0.64,3271,0.64
Reconnaissance,11189,0.55,2798,0.55
Analysis,2142,0.11,535,0.11
Backdoor,1436,0.07,359,0.07
Shellcode,1209,0.06,302,0.06


In [14]:
# save processed data locally
X_train.to_csv(PROCESSED_DIR / "X_train_baseline.csv", index=False)
X_test.to_csv(PROCESSED_DIR / "X_test_baseline.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "y_train_baseline.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test_baseline.csv", index=False)

print("Saved baseline train/test files to:", PROCESSED_DIR.resolve())

Saved baseline train/test files to: C:\Projects\nid-data-mining\data\processed


In [15]:
# Metadata of preprocessing
metadata = {
    "task": "multiclass classification",
    "target": target_col,
    "test_size": 0.2,
    "random_state": 42,
    "stratify": "attack_cat",
    "dropped_columns": ", ".join(drop_cols),
    "categorical_columns": ", ".join(categorical_cols),
    "numeric_columns_count": len(numeric_cols),
    "train_rows": len(X_train),
    "test_rows": len(X_test)
}

metadata_df = pd.DataFrame([metadata])
metadata_df.to_csv(PROCESSED_DIR / "baseline_preprocessing_metadata.csv", index=False)

metadata_df

,task,target,test_size,random_state,stratify,dropped_columns,categorical_columns,numeric_columns_count,train_rows,test_rows
0,multiclass classification,attack_cat,0.2,42,attack_cat,"label, attack_cat, srcip, dstip, stime, ltime","sport, dsport, proto, state, service, ct_ftp_cmd",37,2032037,508010


## Baseline Preprocessing Summary

The four raw UNSW-NB15 CSV files were combined and assigned feature names using `NUSW-NB15_features.csv`.

Baseline preprocessing decisions:
- Binary target variable: `label`
- Dropped `attack_cat` to prevent data leakage
- Dropped `srcip` and `dstip` to reduce dataset-specific memorization
- Dropped timestamp columns if present
- Replaced unidentified service value `-` with `unknown`
- Filled categorical missing values with `unknown`
- Filled numeric missing values with median
- Created an 80/20 stratified train/test split with `random_state=42`

The processed train/test files were saved locally in `data/processed/`.